In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/aisehack-theme-1/data/label/20240529_EO4_RES2_fl_pid_074_label.tif
/kaggle/input/competitions/aisehack-theme-1/data/label/20240529_EO4_RES2_fl_pid_070_label.tif
/kaggle/input/competitions/aisehack-theme-1/data/label/20240529_EO4_RES2_fl_pid_035_label.tif
/kaggle/input/competitions/aisehack-theme-1/data/label/20240529_EO4_RES2_fl_pid_042_label.tif
/kaggle/input/competitions/aisehack-theme-1/data/label/20240529_EO4_RES2_fl_pid_012_label.tif
/kaggle/input/competitions/aisehack-theme-1/data/label/20240529_EO4_RES2_fl_pid_043_label.tif
/kaggle/input/competitions/aisehack-theme-1/data/label/20240529_EO4_RES2_fl_pid_068_label.tif
/kaggle/input/competitions/aisehack-theme-1/data/label/20240529_EO4_RES2_fl_pid_023_label.tif
/kaggle/input/competitions/aisehack-theme-1/data/label/20240529_EO4_RES2_fl_pid_078_label.tif
/kaggle/input/competitions/aisehack-theme-1/data/label/20240529_EO4_RES2_fl_pid_010_label.tif
/kaggle/input/competitions/aisehack-theme-1/data/label/20240

In [2]:
# ── CELL 1 — Install dependencies ─────────────────────────────────────────────
!pip install -q segmentation-models-pytorch albumentations rasterio timm pysheds scipy
print('✅ Libraries ready.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.6 MB/s eta 0:00:00
✅ Libraries ready.


In [3]:
# ── CELL 2 — Imports & global seed ────────────────────────────────────────────
import os, gc, random, warnings, math
from pathlib import Path
import urllib.request
import struct

import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.transform import from_bounds
from scipy.ndimage import sobel, uniform_filter
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, KFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | SMP {smp.__version__} | device: {DEVICE}')

PyTorch 2.9.0+cu126 | SMP 0.5.0 | device: cuda


In [4]:
# ── CELL 3 — Paths & hyperparameters ──────────────────────────────────────────
BASE         = Path('/kaggle/input/competitions/aisehack-theme-1/data')
IMAGE_DIR    = BASE / 'image'
LABEL_DIR    = BASE / 'label'
PRED_IMG_DIR = BASE / 'prediction' / 'image'
OUT_DIR      = Path('/kaggle/working')
GEO_DIR      = OUT_DIR / 'geodata'     # downloaded auxiliary layers stored here
OUT_DIR.mkdir(exist_ok=True)
GEO_DIR.mkdir(exist_ok=True)

# ── Model ────────────────────────────────────────────────────────────────────
ARCH        = 'unetpp'
ENCODER     = 'resnet34'
IN_CHANNELS = 17          # 10 spectral/index + 7 geospatial
IMG_SIZE    = 512

# ── Training ─────────────────────────────────────────────────────────────────
BATCH_SIZE   = 4
NUM_EPOCHS   = 80
LR_ENCODER   = 1e-5
LR_DECODER   = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE     = 20
NUM_WORKERS  = 0

# ── K-Fold ───────────────────────────────────────────────────────────────────
USE_KFOLD   = True
N_FOLDS     = 5

# ── Post-processing ──────────────────────────────────────────────────────────
MIN_COMPONENT_SIZE = 500
MAX_HOLE_SIZE      = 1000

print(f'IN_CHANNELS={IN_CHANNELS} | ENCODER={ENCODER} | ARCH={ARCH}')

IN_CHANNELS=17 | ENCODER=resnet34 | ARCH=unetpp


In [5]:
# ── CELL 4 — Discover AOI bounding box from competition tiles ─────────────────
# We read the geotransform of every training tile and compute the union bounding
# box. This drives all external data downloads — no hardcoded coordinates.

def ids_from_folder(folder, suffix='_image.tif'):
    return sorted(f.replace(suffix, '')
                  for f in os.listdir(folder) if f.endswith(suffix))

all_train_ids = ids_from_folder(IMAGE_DIR)
test_ids      = ids_from_folder(PRED_IMG_DIR)

# Collect tile bounding boxes
all_ids_for_aoi = all_train_ids + test_ids
lon_mins, lat_mins, lon_maxs, lat_maxs = [], [], [], []

for sid in all_ids_for_aoi:
    p = IMAGE_DIR / f'{sid}_image.tif'
    if not p.exists():
        p = PRED_IMG_DIR / f'{sid}_image.tif'
    with rasterio.open(p) as src:
        b = src.bounds
        # Reproject to WGS84 lon/lat if needed
        if src.crs and src.crs.to_epsg() != 4326:
            from rasterio.warp import transform_bounds
            b = transform_bounds(src.crs, 'EPSG:4326', b.left, b.bottom, b.right, b.top)
            b = rasterio.coords.BoundingBox(*b)
        lon_mins.append(b.left);  lat_mins.append(b.bottom)
        lon_maxs.append(b.right); lat_maxs.append(b.top)

# Add 0.05° buffer around union bounding box
BUF = 0.05
AOI = (
    min(lon_mins) - BUF,   # lon_min (west)
    min(lat_mins) - BUF,   # lat_min (south)
    max(lon_maxs) + BUF,   # lon_max (east)
    max(lat_maxs) + BUF,   # lat_max (north)
)
print(f'Train IDs: {len(all_train_ids)}  Test IDs: {len(test_ids)}')
print(f'AOI (W,S,E,N): {[round(x,4) for x in AOI]}')


# ── Load IDs from official split files ──────────────────────────────────────
SPLIT_DIR = BASE / 'split'

def load_split_ids(fname):
    with open(SPLIT_DIR / fname, 'r') as f:
        return [line.strip() for line in f if line.strip()]

train_ids = load_split_ids('train.txt')
val_ids   = load_split_ids('val.txt')
test_ids  = load_split_ids('pred.txt') # or 'pred.txt' depends on competition

all_train_ids = train_ids + val_ids  # combined for k-fold if needed

print(f'Official Split -> Train: {len(train_ids)} | Val: {len(val_ids)} | Test: {len(test_ids)}')


Train IDs: 79  Test IDs: 19
AOI (W,S,E,N): [88.0929, 21.5458, 89.0522, 22.8488]
Official Split -> Train: 59 | Val: 10 | Test: 19


In [6]:
# ── CELL 5 — Download SRTM DEM (NASA, hosted on AWS Open Data) ────────────────
#
# Source  : NASA SRTM GL1 v003 hosted on AWS Open Data Registry (no login needed)
# Bucket  : s3://elevation-tiles-prod/skadi/{lat_band}/{tile}.hgt.gz
# License : Public Domain (NASA data)
# Rule compliance: publicly available, no cost, programmatic download
#
# We compute which 1°×1° SRTM tiles intersect the AOI and download them.

import gzip, shutil

SRTM_BASE = 'https://s3.amazonaws.com/elevation-tiles-prod/skadi'
DEM_DIR   = GEO_DIR / 'srtm'
DEM_DIR.mkdir(exist_ok=True)

def srtm_tile_name(lat, lon):
    """Return SRTM tile filename for floor(lat), floor(lon)."""
    ns = 'N' if lat >= 0 else 'S'
    ew = 'E' if lon >= 0 else 'W'
    return f'{ns}{abs(lat):02d}{ew}{abs(lon):03d}.hgt'

def download_srtm_tiles(aoi):
    """Download all 1°×1° SRTM tiles covering the AOI bounding box."""
    lon_min, lat_min, lon_max, lat_max = aoi
    downloaded = []
    for lat in range(int(math.floor(lat_min)), int(math.ceil(lat_max))):
        lat_band = ('N' if lat >= 0 else 'S') + f'{abs(lat):02d}'
        for lon in range(int(math.floor(lon_min)), int(math.ceil(lon_max))):
            fname = srtm_tile_name(lat, lon)
            out   = DEM_DIR / fname
            if out.exists():
                print(f'  {fname} already exists, skipping.')
                downloaded.append(out)
                continue
            url = f'{SRTM_BASE}/{lat_band}/{fname}.gz'
            gz  = DEM_DIR / (fname + '.gz')
            try:
                print(f'  Downloading {url} ...')
                urllib.request.urlretrieve(url, gz)
                with gzip.open(gz, 'rb') as f_in, open(out, 'wb') as f_out:
                    shutil.copyfileobj(f_in, f_out)
                gz.unlink()
                downloaded.append(out)
                print(f'  ✓ {fname}')
            except Exception as e:
                print(f'  ✗ Failed: {e}')
    return downloaded

srtm_files = download_srtm_tiles(AOI)
print(f'\nDownloaded {len(srtm_files)} SRTM tile(s).')

  ✓ N21E088.hgt
  ✓ N21E089.hgt
  ✓ N22E088.hgt
  ✓ N22E089.hgt

Downloaded 4 SRTM tile(s).


In [7]:
# ── CELL 6 — Download GLO-30 HAND index (Alaska Satellite Facility, AWS Open Data) ──
#
# FIXED: Correct bucket is s3://glo-30-hand (NOT hand-30m.s3.eu-central-1.amazonaws.com)
# Managed by Alaska Satellite Facility (ASF), hosted on AWS us-west-2
# License: CC0 1.0 Universal (public domain — most permissive possible)
# Path structure: https://glo-30-hand.s3.us-west-2.amazonaws.com/v1/2021/{fname}
#
# Tile naming is IDENTICAL to Copernicus GLO-30 DEM tiles:
#   Copernicus_DSM_COG_10_{NS}{lat:02d}_00_{EW}{lon:03d}_00_HAND.tif
# e.g. Copernicus_DSM_COG_10_N21_00_E088_00_HAND.tif

HAND_BASE = 'https://glo-30-hand.s3.us-west-2.amazonaws.com/v1/2021'
HAND_DIR  = GEO_DIR / 'hand'
HAND_DIR.mkdir(exist_ok=True)

def hand_tile_name(lat, lon):
    ns = 'N' if lat >= 0 else 'S'
    ew = 'E' if lon >= 0 else 'W'
    return f'Copernicus_DSM_COG_10_{ns}{abs(lat):02d}_00_{ew}{abs(lon):03d}_00_HAND.tif'

def download_hand_tiles(aoi):
    lon_min, lat_min, lon_max, lat_max = aoi
    downloaded = []
    for lat in range(int(math.floor(lat_min)), int(math.ceil(lat_max))):
        for lon in range(int(math.floor(lon_min)), int(math.ceil(lon_max))):
            fname = hand_tile_name(lat, lon)
            out   = HAND_DIR / fname
            if out.exists():
                print(f'  {fname} already exists.')
                downloaded.append(out)
                continue
            # FIXED URL: correct bucket + versioned path
            url = f'{HAND_BASE}/{fname}'
            try:
                print(f'  Downloading: {url}')
                urllib.request.urlretrieve(url, out)
                downloaded.append(out)
                print(f'  ✓ {fname}  ({out.stat().st_size / 1e6:.1f} MB)')
            except Exception as e:
                print(f'  ✗ {fname}: {e}')
                # Fallback: try the HTTPS endpoint for us-east-1 mirror
                url_fallback = f'https://glo-30-hand.s3.amazonaws.com/v1/2021/{fname}'
                try:
                    print(f'  Trying fallback: {url_fallback}')
                    urllib.request.urlretrieve(url_fallback, out)
                    downloaded.append(out)
                    print(f'  ✓ {fname} (via fallback)')
                except Exception as e2:
                    print(f'  ✗ Fallback also failed: {e2}')
                    # If this tile genuinely has no HAND coverage (e.g. ocean border),
                    # the aux cache builder already handles missing files gracefully
                    # by filling with zeros — no action needed here.
    return downloaded

hand_files = download_hand_tiles(AOI)
print(f'\nDownloaded {len(hand_files)} HAND tile(s).')

  Downloading: https://glo-30-hand.s3.us-west-2.amazonaws.com/v1/2021/Copernicus_DSM_COG_10_N21_00_E088_00_HAND.tif
  ✓ Copernicus_DSM_COG_10_N21_00_E088_00_HAND.tif  (32.0 MB)
  Downloading: https://glo-30-hand.s3.us-west-2.amazonaws.com/v1/2021/Copernicus_DSM_COG_10_N21_00_E089_00_HAND.tif
  ✓ Copernicus_DSM_COG_10_N21_00_E089_00_HAND.tif  (22.9 MB)
  Downloading: https://glo-30-hand.s3.us-west-2.amazonaws.com/v1/2021/Copernicus_DSM_COG_10_N22_00_E088_00_HAND.tif
  ✓ Copernicus_DSM_COG_10_N22_00_E088_00_HAND.tif  (59.5 MB)
  Downloading: https://glo-30-hand.s3.us-west-2.amazonaws.com/v1/2021/Copernicus_DSM_COG_10_N22_00_E089_00_HAND.tif
  ✓ Copernicus_DSM_COG_10_N22_00_E089_00_HAND.tif  (59.3 MB)

Downloaded 4 HAND tile(s).


In [8]:
# ── CELL 7 — Download JRC Global Surface Water occurrence (FINAL FIX) ─────────
import math, shutil, urllib.request
import rasterio

JRC_DIR = GEO_DIR / 'jrc'
JRC_DIR.mkdir(exist_ok=True)

# Clean up any wrong tiles from previous runs
for f in list(JRC_DIR.glob('*.tif')):
    try:
        with rasterio.open(f) as src:
            if src.bounds.top <= 21.0:   # wrong tile — covers south of AOI
                f.unlink()
                print(f'🗑️  Removed wrong tile: {f.name}')
    except Exception:
        f.unlink()

# Your AOI is 21.5–22.8°N → correct JRC tile is 80E_30N (covers 20–30°N)
# Try both v1.1 and v1.4 naming for the 80E_30N tile
JRC_URLS = [
    ('occurrence_80E_30N_v1_1.tif',
     'https://storage.googleapis.com/global-surface-water/downloads2/occurrence/occurrence_80E_30N_v1_1.tif'),
    ('occurrence_80E_30N_v1_4_2021.tif',
     'https://storage.googleapis.com/global-surface-water/downloads2021/occurrence/occurrence_80E_30N_v1_4_2021.tif'),
]

def validate_geotiff(path):
    try:
        with rasterio.open(path) as src:
            _ = src.read(1, window=rasterio.windows.Window(0, 0, 10, 10))
        return True
    except Exception:
        return False

jrc_files = []
for fname, url in JRC_URLS:
    out = JRC_DIR / fname
    if out.exists() and out.stat().st_size > 10_000 and validate_geotiff(out):
        print(f'✓ {fname} already cached ({out.stat().st_size/1e6:.1f} MB)')
        jrc_files.append(out)
        break
    try:
        print(f'Trying: {url}')
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=90) as resp, open(out, 'wb') as f:
            shutil.copyfileobj(resp, f)
        if out.stat().st_size > 10_000 and validate_geotiff(out):
            print(f'✓ {fname}  ({out.stat().st_size/1e6:.1f} MB)')
            jrc_files.append(out)
            break
        else:
            out.unlink()
    except Exception as e:
        print(f'  ✗ {e}')
        if out.exists(): out.unlink()

print(f'\nDownloaded {len(jrc_files)} JRC tile(s).')
if jrc_files:
    with rasterio.open(jrc_files[0]) as src:
        print(f'  Bounds: {src.bounds}')   # must show top=30.0 ✅
        print(f'  Shape : {src.shape}')

Trying: https://storage.googleapis.com/global-surface-water/downloads2/occurrence/occurrence_80E_30N_v1_1.tif
✓ occurrence_80E_30N_v1_1.tif  (72.7 MB)

Downloaded 1 JRC tile(s).
  Bounds: BoundingBox(left=80.0, bottom=20.0, right=90.0, top=30.0)
  Shape : (40000, 40000)


In [9]:
# ── CELL 8 — Download ESA WorldCover 2021 LULC (AWS S3, no-sign-request) ──────
#
# Source  : ESA WorldCover 2021 v200
# Bucket  : s3://esa-worldcover/v200/2021/map/  (public, no credentials)
# URL     : https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/
#           Tile naming: ESA_WorldCover_10m_2021_v200_{lat3}{lon3}_Map.tif
#           e.g. ESA_WorldCover_10m_2021_v200_N21E088_Map.tif  (3°×3° tiles)
# License : CC-BY-4.0 (free, no restrictions)
#
# Class map (used as a float channel after /100 normalisation):
#   10=Tree, 20=Shrub, 30=Grass, 40=Cropland, 50=Built-up,
#   60=Bare, 70=Snow, 80=Water, 90=Wetland, 95=Mangrove, 100=Moss

WC_BASE = 'https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map'
WC_DIR  = GEO_DIR / 'worldcover'
WC_DIR.mkdir(exist_ok=True)

def wc_tile_name(lat, lon):
    """WorldCover tiles are 3°×3°, named by SW corner rounded to 3°."""
    lat3 = int(math.floor(lat / 3)) * 3
    lon3 = int(math.floor(lon / 3)) * 3
    ns   = 'N' if lat3 >= 0 else 'S'
    ew   = 'E' if lon3 >= 0 else 'W'
    return f'ESA_WorldCover_10m_2021_v200_{ns}{abs(lat3):02d}{ew}{abs(lon3):03d}_Map.tif'

def download_wc_tiles(aoi):
    lon_min, lat_min, lon_max, lat_max = aoi
    downloaded = []
    seen = set()
    for lat in range(int(math.floor(lat_min)), int(math.ceil(lat_max))):
        for lon in range(int(math.floor(lon_min)), int(math.ceil(lon_max))):
            fname = wc_tile_name(lat, lon)
            if fname in seen:
                continue
            seen.add(fname)
            out = WC_DIR / fname
            if out.exists():
                print(f'  {fname} already exists.')
                downloaded.append(out); continue
            url = f'{WC_BASE}/{fname}'
            try:
                print(f'  Downloading WorldCover tile: {fname}')
                urllib.request.urlretrieve(url, out)
                downloaded.append(out)
                print(f'  ✓ {fname}')
            except Exception as e:
                print(f'  ✗ {fname}: {e}')
    return downloaded

wc_files = download_wc_tiles(AOI)
print(f'\nDownloaded {len(wc_files)} WorldCover tile(s).')

  ✓ ESA_WorldCover_10m_2021_v200_N21E087_Map.tif

Downloaded 1 WorldCover tile(s).


In [10]:
# ── CELL 9 — Auxiliary layer loader: resample any GeoTIFF to a tile's grid ────
#
# Core utility: reads any source GeoTIFF and resamples it to match the
# exact pixel grid (CRS, transform, shape) of a competition tile.
# Uses bilinear interpolation for continuous data, nearest-neighbour for
# categorical data (LULC class labels).

def resample_to_tile(src_paths, tile_path, band=1,
                     resampling=Resampling.bilinear,
                     nodata_fill=0.0):
    """
    Read one band from src_paths (list of source GeoTIFFs that tile together)
    and resample to exactly match tile_path's grid.

    Returns: (H, W) float32 array aligned to tile_path.
    """
    with rasterio.open(tile_path) as ref:
        dst_crs       = ref.crs
        dst_transform = ref.transform
        dst_width     = ref.width
        dst_height    = ref.height

    out = np.full((dst_height, dst_width), nodata_fill, dtype=np.float32)

    for src_path in src_paths:
        if not Path(src_path).exists():
            continue
        try:
            with rasterio.open(src_path) as src:
                src_data = src.read(band).astype(np.float32)
                # Replace nodata values
                if src.nodata is not None:
                    src_data[src_data == src.nodata] = nodata_fill
                reproject(
                    source      = src_data,
                    destination = out,
                    src_transform = src.transform,
                    src_crs       = src.crs,
                    dst_transform = dst_transform,
                    dst_crs       = dst_crs,
                    resampling    = resampling,
                )
        except Exception as e:
            pass  # tile doesn't cover this area — leave as nodata_fill

    return out  # (H, W) float32


def compute_dem_derivatives(dem):
    """
    Given a (H,W) DEM array, return slope, aspect, and TWI.
    All return values are (H,W) float32.

    Slope  : gradient magnitude in degrees (0–90)
    Aspect : gradient direction in degrees (0–360)
    TWI    : ln(upslope_contrib / tan(slope)) — higher = wetter/flatter
    """
    dx = sobel(dem, axis=1).astype(np.float32)
    dy = sobel(dem, axis=0).astype(np.float32)

    eps   = 1e-6
    grad  = np.sqrt(dx**2 + dy**2)
    slope = np.degrees(np.arctan(grad + eps)).astype(np.float32)    # 0–90°
    aspect= (np.degrees(np.arctan2(-dy, dx)) % 360).astype(np.float32)

    # TWI approximation using local flow accumulation proxy (uniform_filter)
    # Proper TWI needs watershed routing; this is a fast approximation
    slope_rad   = np.radians(slope) + eps
    upslope     = uniform_filter(np.ones_like(dem, np.float32), size=5)  # proxy
    twi         = np.log((upslope + eps) / (np.tan(slope_rad))).astype(np.float32)
    twi         = np.clip(twi, -5, 15)   # reasonable physical range

    return slope, aspect, twi


print('Auxiliary layer utilities defined.')

Auxiliary layer utilities defined.


In [11]:
# ── CELL 9B — DEM Derivative Functions (slope, aspect, TWI) ──────────────────
import numpy as np
from scipy.ndimage import sobel, uniform_filter, generic_filter

def compute_dem_derivatives(dem):
    """
    Compute slope, aspect, and TWI from a DEM array (H, W) float32.
    Returns: slope (H,W), aspect (H,W), twi (H,W) — all float32
    """
    dem = dem.astype(np.float32)

    # Sobel gradients (pixel-scale, unitless)
    dx = sobel(dem, axis=1).astype(np.float32)   # East-West
    dy = sobel(dem, axis=0).astype(np.float32)   # North-South

    # Slope magnitude (rise/run)
    slope = np.sqrt(dx**2 + dy**2)

    # Aspect in radians [-π, π]
    aspect = np.arctan2(dy, dx).astype(np.float32)

    # TWI = ln(A / tan(β))  where A = upslope area proxy, β = slope angle
    slope_rad = np.arctan(slope + 1e-6)                    # avoid tan(0)
    flow_acc  = uniform_filter(slope + 1.0, size=7)        # proxy for flow accumulation
    twi       = np.log(flow_acc / (np.tan(slope_rad) + 1e-6)).astype(np.float32)

    # Clip extreme TWI values (wetlands can blow up to inf)
    twi = np.clip(twi, -10.0, 25.0)

    return slope, aspect, twi


def normalize_band(arr, low=2, high=98):
    """Percentile-clip and normalize array to [0, 1]."""
    lo, hi = np.percentile(arr[np.isfinite(arr)], [low, high])
    arr = np.clip(arr, lo, hi)
    if hi > lo:
        arr = (arr - lo) / (hi - lo)
    return arr.astype(np.float32)


print("✓ compute_dem_derivatives and normalize_band defined.")

✓ compute_dem_derivatives and normalize_band defined.


In [12]:
# ── CELL 10 — Pre-cache all auxiliary layers per tile (FINAL FIX) ─────────────
from scipy.ndimage import sobel, uniform_filter
from rasterio.enums import Resampling
from rasterio.transform import from_bounds
import rasterio.crs

AUX_DIR = OUT_DIR / 'aux_cache'
AUX_DIR.mkdir(exist_ok=True)

# ── normalize to [0,1] via percentile clip ────────────────────────────────────
def normalize_band(arr, low=2, high=98):
    arr = arr.astype(np.float32)
    # Mask out SRTM nodata (-32768) and other invalid values
    arr = np.where(arr < -1000, np.nan, arr)
    arr = np.where(arr > 9000, np.nan, arr)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0 or finite.std() < 1e-6:
        return np.zeros_like(arr)
    lo, hi = np.percentile(finite, [low, high])
    arr = np.clip(arr, lo, hi)
    if hi > lo:
        arr = (arr - lo) / (hi - lo)
    arr = np.nan_to_num(arr, nan=0.0)
    return arr.astype(np.float32)

# ── DEM → slope, aspect, TWI ──────────────────────────────────────────────────
def compute_dem_derivatives(dem):
    dem = dem.astype(np.float32)
    dem = np.nan_to_num(dem, nan=0.0)
    dx  = sobel(dem, axis=1).astype(np.float32)
    dy  = sobel(dem, axis=0).astype(np.float32)
    slope  = np.sqrt(dx**2 + dy**2)
    aspect = np.arctan2(dy, dx).astype(np.float32)
    slope_rad = np.arctan(slope + 1e-6)
    flow_acc  = uniform_filter(slope + 1.0, size=7)
    twi       = np.log(flow_acc / (np.tan(slope_rad) + 1e-6)).astype(np.float32)
    twi       = np.clip(twi, -10.0, 25.0)
    return slope, aspect, twi

# ── resample .hgt with explicit CRS injection ────────────────────────────────
def read_hgt_with_crs(hgt_path):
    """
    SRTM .hgt files are 1°×1° tiles at 3601×3601 resolution.
    Parse tile name to inject correct WGS84 bounds.
    """
    name = Path(hgt_path).stem  # e.g. N21E088
    ns   = name[0]
    lat  = int(name[1:3])
    ew   = name[3]
    lon  = int(name[4:7])

    if ns == 'S': lat = -lat
    if ew == 'W': lon = -lon

    with rasterio.open(hgt_path) as src:
        data = src.read(1).astype(np.float32)
        H, W = data.shape

    # Inject correct geotransform (SW corner = lat, lon; tile = 1°×1°)
    transform = from_bounds(lon, lat, lon + 1, lat + 1, W, H)
    profile = {
        'driver': 'GTiff', 'dtype': 'float32',
        'width': W, 'height': H, 'count': 1,
        'crs': rasterio.crs.CRS.from_epsg(4326),
        'transform': transform, 'nodata': -32768.0
    }
    # Write to a temp .tif with correct CRS for resampling
    tmp_path = Path('/kaggle/working') / f'_tmp_{Path(hgt_path).stem}.tif'
    with rasterio.open(tmp_path, 'w', **profile) as dst:
        dst.write(data, 1)
    return tmp_path

# ── Convert all .hgt to .tif with correct CRS (run once) ─────────────────────
DEM_TIF_DIR = OUT_DIR / 'dem_tif'
DEM_TIF_DIR.mkdir(exist_ok=True)

hgt_files = list(DEM_DIR.glob('*.hgt'))
print(f'Converting {len(hgt_files)} SRTM .hgt tiles to CRS-tagged .tif...')
for hgt in hgt_files:
    out_tif = DEM_TIF_DIR / (hgt.stem + '.tif')
    if out_tif.exists():
        print(f'  ✓ {out_tif.name} already converted')
        continue
    tmp = read_hgt_with_crs(hgt)
    tmp.rename(out_tif)
    print(f'  ✓ Converted {hgt.name} → {out_tif.name}')

dem_tif_files = list(DEM_TIF_DIR.glob('*.tif'))
print(f'DEM .tif files ready: {len(dem_tif_files)}')

# ── get image path for any tile ───────────────────────────────────────────────
def get_tile_path(sid):
    p = IMAGE_DIR / f'{sid}_image.tif'
    if p.exists():
        return p
    return PRED_IMG_DIR / f'{sid}_image.tif'

# ── build 7-channel aux stack ─────────────────────────────────────────────────
def build_aux_stack(sid):
    tile_path = get_tile_path(sid)

    # ── DEM + derivatives (channels 0,1,2,6) ─────────────────────────────────
    if dem_tif_files:
        dem = resample_to_tile(dem_tif_files, tile_path, resampling=Resampling.bilinear)
        dem_norm = normalize_band(dem)
        slope, aspect, twi = compute_dem_derivatives(dem)
        slope_norm  = normalize_band(slope)
        aspect_norm = normalize_band(aspect)
        twi_norm    = normalize_band(twi)
    else:
        with rasterio.open(tile_path) as src:
            H, W = src.height, src.width
        dem_norm = slope_norm = aspect_norm = twi_norm = np.zeros((H, W), np.float32)
        print(f'  ⚠️  [{sid}] No DEM tif files — channels 0,1,2,6 = zero')

    # ── HAND (channel 3) ──────────────────────────────────────────────────────
    hand_files = list(HAND_DIR.glob('*.tif'))
    if hand_files:
        hand = resample_to_tile(hand_files, tile_path,
                                resampling=Resampling.bilinear, nodata_fill=0.0)
        hand_norm = normalize_band(hand)
    else:
        hand_norm = np.zeros_like(dem_norm)
        print(f'  ⚠️  [{sid}] No HAND files — channel 3 = zero')

    # ── JRC water occurrence (channel 4) ──────────────────────────────────────
    jrc_files_local = list(JRC_DIR.glob('*.tif'))
    if jrc_files_local:
        jrc = resample_to_tile(jrc_files_local, tile_path,
                               resampling=Resampling.bilinear, nodata_fill=0.0)
        jrc_norm = normalize_band(jrc)
    else:
        jrc_norm = np.zeros_like(dem_norm)
        print(f'  ⚠️  [{sid}] No JRC files — channel 4 = zero')

    # ── ESA WorldCover LULC (channel 5) ───────────────────────────────────────
    wc_files = list(WC_DIR.glob('*.tif'))
    if wc_files:
        lulc = resample_to_tile(wc_files, tile_path,
                                resampling=Resampling.nearest, nodata_fill=0.0)
        lulc_norm = normalize_band(lulc)
    else:
        lulc_norm = np.zeros_like(dem_norm)
        print(f'  ⚠️  [{sid}] No WorldCover files — channel 5 = zero')

    stack = np.stack([dem_norm, slope_norm, aspect_norm,
                      hand_norm, jrc_norm, lulc_norm, twi_norm], axis=-1)
    return stack.astype(np.float32)

# ── Delete old cache and rebuild ─────────────────────────────────────────────
print('\nClearing old aux cache...')
for f in AUX_DIR.glob('*_aux.npy'):
    f.unlink()

all_sids = sorted(set(all_train_ids + test_ids))
print(f'Pre-caching auxiliary layers for {len(all_sids)} tiles...\n')

for sid in tqdm(all_sids, desc='Aux cache'):
    out_path = AUX_DIR / f'{sid}_aux.npy'
    try:
        aux = build_aux_stack(sid)
        np.save(out_path, aux)
    except Exception as e:
        print(f'  ⚠️  [{sid}] Failed: {e} — saving zeros.')
        np.save(out_path, np.zeros((IMG_SIZE, IMG_SIZE, 7), np.float32))

print('\n✓ Auxiliary cache complete.')
print(f'  Cached {len(list(AUX_DIR.glob("*_aux.npy")))} files in {AUX_DIR}')

# ── Diagnostic ────────────────────────────────────────────────────────────────
print('\nPer-channel quality check (zero% > 80% = problem):')
ch_names = ['DEM', 'Slope', 'Aspect', 'HAND', 'JRC', 'LULC', 'TWI']
channel_zeros = np.zeros(7)
n_checked = 0
for sid in all_sids[:10]:
    arr = np.load(AUX_DIR / f'{sid}_aux.npy')
    for i in range(7):
        channel_zeros[i] += (arr[..., i] == 0).mean()
    n_checked += 1

for i, name in enumerate(ch_names):
    pct = 100 * channel_zeros[i] / n_checked
    flag = '✅' if pct < 80 else '❌ PROBLEM'
    print(f'  [{i}] {name:8s}  avg_zero={pct:.1f}%  {flag}')

Converting 4 SRTM .hgt tiles to CRS-tagged .tif...
  ✓ Converted N22E088.hgt → N22E088.tif
  ✓ Converted N22E089.hgt → N22E089.tif
  ✓ Converted N21E089.hgt → N21E089.tif
  ✓ Converted N21E088.hgt → N21E088.tif
DEM .tif files ready: 4

Clearing old aux cache...
Pre-caching auxiliary layers for 88 tiles...



Aux cache:   0%|          | 0/88 [00:00<?, ?it/s]


✓ Auxiliary cache complete.
  Cached 88 files in /kaggle/working/aux_cache

Per-channel quality check (zero% > 80% = problem):
  [0] DEM       avg_zero=100.0%  ❌ PROBLEM
  [1] Slope     avg_zero=100.0%  ❌ PROBLEM
  [2] Aspect    avg_zero=100.0%  ❌ PROBLEM
  [3] HAND      avg_zero=100.0%  ❌ PROBLEM
  [4] JRC       avg_zero=51.7%  ✅
  [5] LULC      avg_zero=12.9%  ✅
  [6] TWI       avg_zero=0.0%  ✅


In [13]:
# ── CELL 11 — Feature engineering (spectral bands + indices) ─────────────────
def load_raw_image(path):
    """Load 6-band .tif → (H, W, 10) float32 with spectral indices."""
    with rasterio.open(path) as src:
        img = src.read().astype(np.float32)   # (6, H, W)
    green, red, nir, swir = img[2], img[3], img[4], img[5]
    eps   = 1e-8
    ndwi  = (green - nir)        / (green + nir        + eps)
    ndvi  = (nir   - red)        / (nir   + red        + eps)
    mndwi = (green - swir)       / (green + swir       + eps)
    ewi   = (green - swir - nir) / (green + swir + nir + eps)
    out   = np.concatenate(
        [img, ndwi[None], ndvi[None], mndwi[None], ewi[None]], axis=0
    )
    return out.transpose(1, 2, 0)   # (H, W, 10)

def load_mask(path):
    with rasterio.open(path) as src:
        return src.read(1).astype(np.float32)

print('Feature engineering defined (10 spectral/index channels).')

Feature engineering defined (10 spectral/index channels).


In [14]:
# ── CELL 12 — Deterministic channel normalization stats ───────────────────────

# ── Safety: define AUX_DIR if Cell 10 hasn't run yet ─────────────────────────
from pathlib import Path
if 'AUX_DIR' not in dir():
    AUX_DIR = Path('/kaggle/working') / 'aux_cache'
    AUX_DIR.mkdir(exist_ok=True)
    print(f'⚠️  AUX_DIR was not defined — set to {AUX_DIR}')
    print(f'   Make sure Cell 10 (aux cache builder) has been run first!')

# ── Also define load_raw_image if missing ─────────────────────────────────────
if 'load_raw_image' not in dir():
    def load_raw_image(path):
        with rasterio.open(path) as src:
            arr = src.read().astype(np.float32)   # (C, H, W)
        return np.transpose(arr, (1, 2, 0))        # → (H, W, C)
    print('⚠️  load_raw_image was not defined — defined inline.')

def compute_channel_stats(ids, image_dir, aux_dir, n_ch=IN_CHANNELS):
    accum = [[] for _ in range(n_ch)]

    for sid in tqdm(sorted(ids), desc='Computing stats'):
        # Spectral (10 channels)
        spec = load_raw_image(image_dir / f'{sid}_image.tif')
        for c in range(10):
            ch = spec[:, :, c].ravel()
            lo, hi = np.percentile(ch, [2, 98])
            accum[c].append(np.clip(ch, lo, hi))

        # Geospatial (7 channels)
        aux_path = aux_dir / f'{sid}_aux.npy'
        if not aux_path.exists():
            print(f'  ⚠️  Missing aux cache for {sid} — skipping.')
            continue
        aux = np.load(aux_path)   # (H, W, 7)
        for c in range(7):
            ch = aux[:, :, c].ravel()
            lo, hi = np.percentile(ch, [2, 98])
            accum[10 + c].append(np.clip(ch, lo, hi))

    means = np.array([np.concatenate(b).mean() if b else 0.0 for b in accum], np.float32)
    stds  = np.array([np.concatenate(b).std()  if b else 1.0 for b in accum], np.float32)
    stds  = np.where(stds < 1e-8, 1.0, stds)
    return means, stds

CHAN_MEAN, CHAN_STD = compute_channel_stats(train_ids, IMAGE_DIR, AUX_DIR)
print('Channel means (first 10 spectral, last 7 geospatial):')
print('  means:', np.round(CHAN_MEAN, 3))
print('  stds :', np.round(CHAN_STD,  3))

Computing stats:   0%|          | 0/59 [00:00<?, ?it/s]

Channel means (first 10 spectral, last 7 geospatial):
  means: [ 7.798920e+02  3.568020e+02  1.980044e+03  1.827385e+03  1.939811e+03
  1.392769e+03  8.000000e-03  3.700000e-02  1.890000e-01 -2.510000e-01
  0.000000e+00  0.000000e+00  0.000000e+00  1.000000e-03  1.830000e-01
  5.560000e-01  1.312200e+01]
  stds : [3.67255e+02 1.58900e+02 6.31325e+02 6.13834e+02 5.90001e+02 5.23412e+02
 3.60000e-02 4.60000e-02 1.03000e-01 5.30000e-02 1.00000e+00 1.00000e+00
 1.00000e+00 3.30000e-02 3.67000e-01 3.86000e-01 1.00000e+00]


In [15]:
# ── CELL 13 — Dataset: 10 spectral + 7 geospatial = 17 channels ──────────────
class FloodDataset(Dataset):
    def __init__(self, ids, image_dir, aux_dir, label_dir=None,
                 transform=None, is_test=False, means=None, stds=None):
        self.ids       = ids
        self.image_dir = Path(image_dir)
        self.aux_dir   = Path(aux_dir)
        self.label_dir = Path(label_dir) if label_dir else None
        self.transform = transform
        self.is_test   = is_test
        self.means = means if means is not None else np.zeros(IN_CHANNELS, np.float32)
        self.stds  = stds  if stds  is not None else np.ones(IN_CHANNELS,  np.float32)
    
    def __len__(self):
        return len(self.ids)
    
    def _normalize(self, img):
        return np.clip((img - self.means) / self.stds, -5.0, 5.0).astype(np.float32)
    
    def __getitem__(self, idx):
        sid = self.ids[idx]
        
        # Spectral (H,W,10) - Check prediction folder first for test data
        img_path = self.image_dir / f'{sid}_image.tif'
        if not img_path.exists():
            # Try prediction folder (for test files 080-098)
            img_path = PRED_IMG_DIR / f'{sid}_image.tif'
        
        spec = load_raw_image(img_path)
        
        # Geospatial (H,W,7)
        aux_path = self.aux_dir / f'{sid}_aux.npy'
        aux = np.load(aux_path) if aux_path.exists() else \
              np.zeros((spec.shape[0], spec.shape[1], 7), np.float32)
        
        # Stack → (H,W,17), normalize
        img = self._normalize(np.concatenate([spec, aux], axis=-1))
        
        if self.is_test:
            if self.transform:
                img = self.transform(image=img)['image']
            return img, sid
        
        mask = load_mask(self.label_dir / f'{sid}_label.tif')
        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        return img, mask.unsqueeze(0).float()

print('FloodDataset (17-channel) defined.')

FloodDataset (17-channel) defined.


In [16]:
# ── CELL 14 — SAR-safe augmentations ─────────────────────────────────────────
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(
        brightness_limit=0.10, contrast_limit=0.10, p=0.30),
    A.MultiplicativeNoise(
        multiplier=(0.90, 1.10), per_channel=True, p=0.30),
    A.CoarseDropout(
        max_holes=6, max_height=32, max_width=32, fill_value=0, p=0.20),
    ToTensorV2(),
])
val_transform = A.Compose([ToTensorV2()])
print('Augmentations defined.')

Augmentations defined.


In [17]:
# ── CELL 15 — Loss, metrics, model factory ───────────────────────────────────
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0): super().__init__(); self.smooth=smooth
    def forward(self, lo, t):
        p=torch.sigmoid(lo).view(-1); t=t.view(-1)
        return 1-(2*(p*t).sum()+self.smooth)/(p.sum()+t.sum()+self.smooth)

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.65, gamma=2.5): super().__init__(); self.a=alpha; self.g=gamma
    def forward(self, lo, t):
        bce=F.binary_cross_entropy_with_logits(lo,t,reduction='none')
        return (self.a*(1-torch.exp(-bce))**self.g*bce).mean()

class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, smooth=1.0):
        super().__init__(); self.a=alpha; self.b=beta; self.s=smooth
    def forward(self, lo, t):
        p=torch.sigmoid(lo).view(-1); t=t.view(-1)
        tp=(p*t).sum(); fp=(p*(1-t)).sum(); fn=((1-p)*t).sum()
        return 1-(tp+self.s)/(tp+self.a*fp+self.b*fn+self.s)

class CombinedLoss(nn.Module):
    def __init__(self): super().__init__(); self.d=DiceLoss(); self.f=FocalLoss(); self.t=TverskyLoss()
    def forward(self, lo, t): return 0.4*self.d(lo,t)+0.3*self.f(lo,t)+0.3*self.t(lo,t)

def batch_iou(lo, t, thr=0.5, eps=1e-8):
    """Modified to return raw intersection/union for Global IoU calculation."""
    p = ((torch.sigmoid(lo) > thr).float()).view(lo.size(0), -1)
    t = t.float().view(t.size(0), -1)
    intersection = (p * t).sum(1)
    union = p.sum(1) + t.sum(1) - intersection
    return {
        'intersection': intersection.sum().item(),
        'union': union.sum().item(),
        'iou': ((intersection + eps) / (union + eps)).mean().item()
    }

def batch_dice(lo, t, thr=0.5, eps=1e-8):
    p=((torch.sigmoid(lo)>thr).float()).view(lo.size(0),-1); t=t.float().view(t.size(0),-1)
    i=(p*t).sum(1)
    return ((2*i+eps)/(p.sum(1)+t.sum(1)+eps)).mean().item()

def build_model():
    kw=dict(encoder_name=ENCODER,encoder_weights='imagenet',in_channels=IN_CHANNELS,
            classes=1,activation=None,decoder_use_batchnorm=True)
    return (smp.UnetPlusPlus(**kw) if ARCH=='unetpp' else
            smp.Unet(**kw) if ARCH=='unet' else smp.DeepLabV3Plus(**kw))

def build_optimizer(model):
    enc_params   = list(model.encoder.parameters())
    other_params = [p for p in model.parameters() if not any(p is e for e in enc_params)]
    return torch.optim.AdamW(
        [{'params':enc_params,'lr':LR_ENCODER},{'params':other_params,'lr':LR_DECODER}],
        weight_decay=WEIGHT_DECAY)

def build_scheduler(opt):
    from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingWarmRestarts
    w=LinearLR(opt,start_factor=0.1,end_factor=1.0,total_iters=5)
    c=CosineAnnealingWarmRestarts(opt,T_0=15,T_mult=2,eta_min=1e-6)
    return SequentialLR(opt,schedulers=[w,c],milestones=[5])

_m=build_model().to(DEVICE)
print(f'Model: {ARCH}+{ENCODER} | params: {sum(p.numel() for p in _m.parameters() if p.requires_grad):,}')
del _m; gc.collect()

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Model: unetpp+resnet34 | params: 26,122,513


40

In [18]:
# ── CELL 16 — Training loop (single fold) ─────────────────────────────────────
def train_one_fold(fold_idx, tr_ids, vl_ids, ckpt_path):
    print(f'\n{"="*60}\nFOLD {fold_idx}  train={len(tr_ids)}  val={len(vl_ids)}\n{"="*60}')

    tr_ds = FloodDataset(tr_ids, IMAGE_DIR, AUX_DIR, LABEL_DIR,
                         transform=train_transform, means=CHAN_MEAN, stds=CHAN_STD)
    vl_ds = FloodDataset(vl_ids, IMAGE_DIR, AUX_DIR, LABEL_DIR,
                         transform=val_transform, means=CHAN_MEAN, stds=CHAN_STD)
    tr_loader = DataLoader(tr_ds,batch_size=BATCH_SIZE,shuffle=True,
                            num_workers=NUM_WORKERS,pin_memory=True,drop_last=True)
    vl_loader = DataLoader(vl_ds,batch_size=BATCH_SIZE,shuffle=False,
                            num_workers=NUM_WORKERS,pin_memory=True)

    model     = build_model().to(DEVICE)
    criterion = CombinedLoss()
    optimizer = build_optimizer(model)
    scheduler = build_scheduler(optimizer)
    scaler    = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=='cuda'))

    best_val_iou, patience_ctr = 0.0, 0

    for epoch in range(1, NUM_EPOCHS+1):
        model.train(); t_loss=0.0
        for imgs,masks in tqdm(tr_loader,desc=f'Ep{epoch:03d} tr',leave=False):
            imgs=imgs.to(DEVICE,dtype=torch.float32)
            masks=masks.to(DEVICE,dtype=torch.float32)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
                loss=criterion(model(imgs),masks)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(optimizer); scaler.update()
            t_loss+=loss.item()
        scheduler.step()

        model.eval(); v_loss = 0.0
        total_inter, total_union, v_dice = 0.0, 0.0, 0.0
        with torch.no_grad():
            for imgs,masks in vl_loader:
                imgs=imgs.to(DEVICE,dtype=torch.float32)
                masks=masks.to(DEVICE,dtype=torch.float32)
                with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
                    lo=model(imgs)
                v_loss+=criterion(lo,masks).item()
                res = batch_iou(lo,masks)
                total_inter += res['intersection']
                total_union += res['union']
                v_dice += batch_dice(lo,masks)

        n=len(vl_loader)
        avg_t=t_loss/len(tr_loader); avg_vl=v_loss/n; 
        avg_vi=(total_inter + 1e-8)/(total_union + 1e-8); avg_vd=v_dice/n

        if avg_vi>best_val_iou:
            best_val_iou=avg_vi
            torch.save({'epoch':epoch,'model':model.state_dict(),'best_iou':best_val_iou},ckpt_path)
            patience_ctr=0; tag='✓ saved'
        else:
            patience_ctr+=1; tag=f'patience {patience_ctr}/{PATIENCE}'

        lr_now=optimizer.param_groups[1]['lr']
        print(f'Ep{epoch:03d} | tr{avg_t:.4f} vl{avg_vl:.4f} IoU{avg_vi:.4f} Dice{avg_vd:.4f} lr{lr_now:.1e} {tag}')
        if patience_ctr>=PATIENCE: print('Early stop.'); break

    print(f'Fold {fold_idx} best IoU: {best_val_iou:.4f}')
    del model,optimizer,scheduler,scaler,tr_ds,vl_ds,tr_loader,vl_loader
    gc.collect(); torch.cuda.empty_cache()
    return best_val_iou

print('Training loop defined.')

Training loop defined.


In [19]:
# ── CELL 17 — 6-pass D4 TTA ──────────────────────────────────────────────────
def tta_predict(model, img_tensor, device):
    model.eval()
    img=img_tensor.to(device,dtype=torch.float32); preds=[]
    with torch.no_grad(),torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
        preds.append(torch.sigmoid(model(img)))
        ih=torch.flip(img,[3]); preds.append(torch.flip(torch.sigmoid(model(ih)),[3]))
        iv=torch.flip(img,[2]); preds.append(torch.flip(torch.sigmoid(model(iv)),[2]))
        ihv=torch.flip(img,[2,3]); preds.append(torch.flip(torch.sigmoid(model(ihv)),[2,3]))
        r1=torch.rot90(img,k=1,dims=[2,3]); preds.append(torch.rot90(torch.sigmoid(model(r1)),k=-1,dims=[2,3]))
        r3=torch.rot90(img,k=3,dims=[2,3]); preds.append(torch.rot90(torch.sigmoid(model(r3)),k=-3,dims=[2,3]))
    return torch.stack(preds).mean(dim=0).cpu()

print('6-pass TTA defined.')
def sharpen(p, power=1.1):
    return p ** power


6-pass TTA defined.


In [20]:
# ── CELL 18 — Val threshold sweep (TTA-consistent) ──────────────────────────
def find_best_threshold(model, vl_ids):
    vl_ds=FloodDataset(vl_ids,IMAGE_DIR,AUX_DIR,LABEL_DIR,
                        transform=val_transform,means=CHAN_MEAN,stds=CHAN_STD)
    vl_loader=DataLoader(vl_ds,batch_size=1,shuffle=False,num_workers=NUM_WORKERS)
    model.eval(); all_p,all_m=[],[]
    with torch.no_grad():
        for imgs,masks in tqdm(vl_loader,desc='Sweep probs',leave=False):
            # Use TTA + same sharpening as inference — threshold must match
            prob = tta_predict(model, imgs, DEVICE)
            all_p.append(sharpen(prob, power=1.1).numpy())
            all_m.append(masks.numpy())
    pf=np.concatenate(all_p).ravel(); mf=np.concatenate(all_m).ravel().astype(np.uint8)
    best_t,best_iou=0.5,0.0
    for t in np.arange(0.1, 0.9, 0.005):
        p=(pf>t).astype(np.uint8)
        iou=(p&mf).sum()/((p|mf).sum()+1e-8)
        if iou>best_iou: best_iou=iou; best_t=t
    print(f'  Best thr={best_t:.3f}  Global IoU={best_iou:.4f}')
    return float(best_t)

print('Threshold sweep defined.')


Threshold sweep defined.


In [21]:
# ── CELL 19 — Morphological post-processing ───────────────────────────────────
from scipy import ndimage

def postprocess(binary, min_size=250, max_hole=500):
    labeled,n=ndimage.label(binary)
    for cid in range(1,n+1):
        if (labeled==cid).sum()<min_size: binary[labeled==cid]=0
    if binary.any():
        inv=1-binary; lh,nh=ndimage.label(inv)
        for hid in range(1,nh+1):
            hole=lh==hid
            if hole[0,:].any() or hole[-1,:].any() or hole[:,0].any() or hole[:,-1].any(): continue
            if hole.sum()<max_hole: binary[hole]=1
    return binary.astype(np.uint8)

def mask_to_rle(mask):
    pixels=mask.flatten(order='F')
    pixels=np.concatenate([[0],pixels,[0]])
    runs=np.where(pixels[1:]!=pixels[:-1])[0]+1
    runs[1::2]-=runs[::2]
    return ' '.join(str(x) for x in runs) if len(runs) else ''

print('Post-processing and RLE defined.')

Post-processing and RLE defined.


In [22]:
# ── CELL 20 — MAIN: K-Fold train + ensemble inference ─────────────────────────
test_ds     = FloodDataset(test_ids, PRED_IMG_DIR, AUX_DIR,
                            transform=val_transform,
                            means=CHAN_MEAN, stds=CHAN_STD, is_test=True)
test_loader = DataLoader(test_ds,batch_size=1,shuffle=False,num_workers=NUM_WORKERS)

prob_accum = {sid: np.zeros((IMG_SIZE,IMG_SIZE),np.float64) for sid in test_ids}
fold_thresholds, fold_best_ious = [], []

if USE_KFOLD:
    kf=KFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)
    folds=[(list(np.array(all_train_ids)[tr]),list(np.array(all_train_ids)[vl]))
           for tr,vl in kf.split(all_train_ids)]
else:
    folds=[(train_ids,val_ids)]

n_models=len(folds)
print(f'Training {n_models} model(s) with {IN_CHANNELS} input channels...')

for fold_idx,(tr_ids,vl_ids) in enumerate(folds,1):
    ckpt_path=OUT_DIR/f'best_fold{fold_idx}.pth'
    best_iou=train_one_fold(fold_idx,tr_ids,vl_ids,ckpt_path)
    fold_best_ious.append(best_iou)

    model=build_model().to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path,map_location=DEVICE)['model'])

    thr=find_best_threshold(model,vl_ids)
    fold_thresholds.append(thr)

    print(f'  TTA inference fold {fold_idx}...')
    for imgs,sids in tqdm(test_loader,desc=f'  TTA f{fold_idx}'):
        prob=tta_predict(model,imgs,DEVICE)
        prob_accum[sids[0]]+=sharpen(prob.squeeze().numpy(), power=1.1)

    del model; gc.collect(); torch.cuda.empty_cache()

THRESHOLD=float(np.median(fold_thresholds))
print(f'\nAll folds done | mean IoU={np.mean(fold_best_ious):.4f} | ensemble thr={THRESHOLD:.3f}')

Training 5 model(s) with 17 input channels...

FOLD 1  train=55  val=14


Ep001 tr:   0%|          | 0/13 [00:00<?, ?it/s]

Ep001 | tr0.4447 vl0.5814 IoU0.3923 Dice0.5132 lr2.8e-05 ✓ saved


Ep002 tr:   0%|          | 0/13 [00:00<?, ?it/s]

Ep002 | tr0.4047 vl0.5559 IoU0.4681 Dice0.5778 lr4.6e-05 ✓ saved


...

Ep076 | tr0.2329 vl0.5552 IoU0.5784 Dice0.5726 lr6.1e-05 patience 20/20
Early stop.
Fold 5 best IoU: 0.5997


Sweep probs:   0%|          | 0/13 [00:00<?, ?it/s]

  Best thr=0.490  Global IoU=0.6058
  TTA inference fold 5...


  TTA f5:   0%|          | 0/19 [00:00<?, ?it/s]


All folds done | mean IoU=0.6305 | ensemble thr=0.490


In [23]:
# ── CELL 21 — Build and save submission.csv ───────────────────────────────────
import pandas as pd

rows = []
for sid in test_ids:
    avg_prob = prob_accum[sid] / n_models
    binary   = (avg_prob > THRESHOLD).astype(np.uint8)
    binary   = postprocess(binary)
    rle      = mask_to_rle(binary)
    # Empty string for no-flood tiles — never append NaN
    rows.append({'id': sid, 'rle_mask': rle if rle else ''})
    pct = 100 * binary.sum() / (IMG_SIZE * IMG_SIZE)
    print(f'  {sid:30s}  flood%={pct:5.1f}  rle_len={len(rle.split()) if rle else 0}')

# BUILD sub_df from rows — this line was missing before
sub_df = pd.DataFrame(rows, columns=['id', 'rle_mask'])

# Save with keep_default_na protection
sub_df.to_csv(OUT_DIR / 'submission.csv', index=False, na_rep='')

# Verify by reloading
verify = pd.read_csv(OUT_DIR / 'submission.csv', keep_default_na=False)
print(f'\n✅ submission.csv saved')
print(f'   Rows            : {len(verify)}')
print(f'   NaN count       : {verify["rle_mask"].isna().sum()}  <-- must be 0')
print(f'   Non-empty masks : {(verify["rle_mask"] != "").sum()}')
print(f'   Empty masks     : {(verify["rle_mask"] == "").sum()}')
verify.head(10)

  20240529_EO4_RES2_fl_pid_080    flood%=  7.4  rle_len=726
  20240529_EO4_RES2_fl_pid_081    flood%= 16.2  rle_len=1716
  20240529_EO4_RES2_fl_pid_082    flood%=  7.6  rle_len=1160
  20240529_EO4_RES2_fl_pid_083    flood%= 28.2  rle_len=2690
  20240529_EO4_RES2_fl_pid_084    flood%= 74.7  rle_len=2274
  20240529_EO4_RES2_fl_pid_085    flood%= 68.2  rle_len=3386
  20240529_EO4_RES2_fl_pid_086    flood%= 28.1  rle_len=2928
  20240529_EO4_RES2_fl_pid_087    flood%=  0.3  rle_len=110
  20240529_EO4_RES2_fl_pid_088    flood%=  8.4  rle_len=508
  20240529_EO4_RES2_fl_pid_089    flood%= 20.1  rle_len=1786
  20240529_EO4_RES2_fl_pid_090    flood%= 19.7  rle_len=2324
  20240529_EO4_RES2_fl_pid_091    flood%=  2.4  rle_len=244
  20240529_EO4_RES2_fl_pid_092    flood%= 23.7  rle_len=2388
  20240529_EO4_RES2_fl_pid_093    flood%=  8.0  rle_len=1724
  20240529_EO4_RES2_fl_pid_094    flood%=  0.8  rle_len=206
  20240529_EO4_RES2_fl_pid_095    flood%=  8.9  rle_len=858
  20240529_EO4_RES2_fl_pid_096

,id,rle_mask
0,20240529_EO4_RES2_fl_pid_080,451 22 963 22 1475 22 1987 22 2498 23 3010 23 ...
1,20240529_EO4_RES2_fl_pid_081,340 16 852 16 1364 16 1876 16 2388 16 2900 16 ...
2,20240529_EO4_RES2_fl_pid_082,149 11 661 11 1172 13 1684 13 2196 13 2708 13 ...
3,20240529_EO4_RES2_fl_pid_083,1 32 47 45 513 32 559 44 1025 32 1071 44 1537 ...
4,20240529_EO4_RES2_fl_pid_084,133 119 293 71 378 135 645 119 805 71 890 135 ...
5,20240529_EO4_RES2_fl_pid_085,123 149 306 26 336 78 447 66 635 149 818 25 84...
6,20240529_EO4_RES2_fl_pid_086,5 17 38 91 139 140 283 88 484 24 517 17 550 91...
7,20240529_EO4_RES2_fl_pid_087,17628 3 18139 5 18651 6 19162 7 19673 8 20185 ...
8,20240529_EO4_RES2_fl_pid_088,25758 3 26268 7 26779 8 27290 9 27802 10 28314...
9,20240529_EO4_RES2_fl_pid_089,41351 3 41861 8 42373 10 42884 12 43397 12 439...
